In [17]:
import pandas as pd

In [19]:
df = pd.read_csv('../data/raw/cleaned_data_v2.csv')

In [20]:
df.head()

,bhk,type,locality,area,price,price_unit,region,status,age,final_price
0,3,Apartment,Lak And Hanware The Residency Tower,685,2.50,Cr,Andheri West,Ready to move,New,250.00
1,2,Apartment,Radheya Sai Enclave Building No 2,640,52.51,L,Naigaon East,Under Construction,New,52.51
2,2,Apartment,Romell Serene,610,1.73,Cr,Borivali West,Under Construction,New,173.00
3,2,Apartment,Soundlines Codename Urban Rainforest,876,59.98,L,Panvel,Under Construction,New,59.98
4,2,Apartment,Origin Oriana,659,94.11,L,Mira Road East,Under Construction,New,94.11


In [22]:
df.shape

(76038, 10)

In [23]:
df.columns.tolist()

['bhk',
 'type',
 'locality',
 'area',
 'price',
 'price_unit',
 'region',
 'status',
 'age',
 'final_price']

In [24]:
df[["price", "price_unit", "final_price"]].head(10)

,price,price_unit,final_price
0,2.50,Cr,250.00
1,52.51,L,52.51
2,1.73,Cr,173.00
3,59.98,L,59.98
4,94.11,L,94.11
5,3.30,Cr,330.00
6,1.99,Cr,199.00
7,2.89,Cr,289.00
8,29.84,L,29.84
9,2.72,Cr,272.00


In [25]:
print(df["price_unit"].value_counts(dropna=False))
print("\nMissing values:")
print(df[["price", "price_unit", "final_price"]].isnull().sum())

price_unit
Cr    40981
L     35057
Name: count, dtype: int64

Missing values:
price          0
price_unit     0
final_price    0
dtype: int64


In [26]:
# Check whether final_price matches price converted to lakhs
expected_price = df["price"].where(
    df["price_unit"].eq("L"),
    df["price"] * 100
)

difference = (df["final_price"] - expected_price).abs()

print("Maximum conversion difference:", difference.max())
print("Rows with inconsistent prices:", (difference > 0.01).sum())
print("\nFinal price summary (lakhs):")
print(df["final_price"].describe().round(2))

Maximum conversion difference: 4.547473508864641e-13
Rows with inconsistent prices: 0

Final price summary (lakhs):
count    76038.00
mean       168.42
std        217.67
min          4.49
25%         64.00
50%        110.00
75%        194.00
max       6000.00
Name: final_price, dtype: float64


In [27]:
df.dtypes

bhk              int64
type               str
locality           str
area             int64
price          float64
price_unit         str
region             str
status             str
age                str
final_price    float64
dtype: object

In [29]:
df.isnull().sum()

bhk            0
type           0
locality       0
area           0
price          0
price_unit     0
region         0
status         0
age            0
final_price    0
dtype: int64

In [32]:
df.duplicated().sum()

np.int64(20312)

In [33]:
print("\nUNIQUE VALUES IN CATEGORICAL COLUMNS")
for col in ["type", "region", "status", "age"]:
    print(f"\n{col}:")
    print(df[col].value_counts().head(10))


UNIQUE VALUES IN CATEGORICAL COLUMNS

type:
type
Apartment            74854
Studio Apartment       882
Villa                  226
Independent House       73
Penthouse                3
Name: count, dtype: int64

region:
region
Thane West        14868
Mira Road East     9902
Dombivali          3041
Kandivali East     2568
Kharghar           2362
Andheri West       1842
Kalyan West        1815
Virar              1557
Mulund West        1526
Panvel             1474
Name: count, dtype: int64

status:
status
Ready to move         44982
Under Construction    31056
Name: count, dtype: int64

age:
age
New        38072
Resale     23357
Unknown    14609
Name: count, dtype: int64


In [34]:
# Features (X): information used to predict price
feature_columns = [
    "bhk",
    "type",
    "locality",
    "area",
    "region",
    "status",
    "age"
]

X = df[feature_columns]
y = df["final_price"]

In [36]:
X.shape

(76038, 7)

In [37]:
y.shape

(76038,)

In [38]:
X.head()

,bhk,type,locality,area,region,status,age
0,3,Apartment,Lak And Hanware The Residency Tower,685,Andheri West,Ready to move,New
1,2,Apartment,Radheya Sai Enclave Building No 2,640,Naigaon East,Under Construction,New
2,2,Apartment,Romell Serene,610,Borivali West,Under Construction,New
3,2,Apartment,Soundlines Codename Urban Rainforest,876,Panvel,Under Construction,New
4,2,Apartment,Origin Oriana,659,Mira Road East,Under Construction,New


In [39]:
y.head()

0    250.00
1     52.51
2    173.00
3     59.98
4     94.11
Name: final_price, dtype: float64

In [40]:
from sklearn.model_selection import train_test_split

In [41]:
X_train, X_test, y_train, y_test = train_test_split (
    X,
    y,
    test_size = 0.2,
    random_state = 42
)

In [45]:
X_train.shape, y_train.shape

((60830, 7), (60830,))

In [46]:
X_test.shape, y_test.shape

((15208, 7), (15208,))

In [47]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer

categorical_features = [
    "type", "locality", "region", "status", "age"
]

numeric_features = ["bhk", "area"]

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", SimpleImputer(strategy="median"), numeric_features),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features)
    ]
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

model.fit(X_train, y_train)

print("Model training complete!")

Model training complete!


In [51]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

y_pred = model.predict(X_test)

In [52]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

In [54]:
print(f"MAE: ₹{mae:.2f} lakhs") 
print(f"MSE: ₹{mse:.2f} lakhs") 
print(f"RMSE: ₹{rmse:.2f} lakhs") 

MAE: ₹35.07 lakhs
MSE: ₹6344.28 lakhs
RMSE: ₹79.65 lakhs


In [55]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error

baseline = DummyRegressor(strategy="median")
baseline.fit(X_train, y_train)

baseline_predictions = baseline.predict(X_test)

baseline_mae = mean_absolute_error(
    y_test, baseline_predictions
)

print(f"Baseline MAE: ₹{baseline_mae:.2f} lakhs")
print(f"Linear Regression MAE: ₹{mae:.2f} lakhs")

Baseline MAE: ₹103.54 lakhs
Linear Regression MAE: ₹35.07 lakhs
